In [ ]:
#!pip install astropy

In [ ]:
import numpy as np
import matplotlib.pylab as plt
from astropy.io import fits

In [ ]:
fn = "/Users/natsuki/Projects/hogg_research/hoggnation/oscillator_catalog/good_parents_fit.fits"
with fits.open(fn) as hdu_list:
    print(hdu_list.info())
    data = hdu_list[1].data
    header = hdu_list[1].header
print(len(data), header)

In [ ]:
features = data["features"]
print(features.shape)

#create the time invariant feature vector for clustering (log(a0^2), log(a1^2 + b1^2), ...)
squared_feats = np.zeros((1438, 33))
for e,f in enumerate(features):
    j = 0
    for i in range(len(f) - 1):
        if i == 0:
            squared_feats[e][j] = f[i]**2
            j = j + 1
        if i%2 == 1:
            squared_feats[e][j] = f[i]**2 + f[i+1]**2
            j = j + 1


## plotting

In [ ]:
#from hogg
foo, k = squared_feats.shape
frequencies = np.outer(data["refined_frequency"], (1. + np.arange(k)))
informations = np.nansum(squared_feats * frequencies * (frequencies < 24.), axis=1)
print(informations)
refine_freqs = data["refined_frequency"]

In [ ]:
#from hogg
sizes = 0.5 * np.log10(informations)
sizes += 4.
sizes = np.clip(sizes, 0.01, None)
print(sizes)
for i, j in [(0, 1),
             (0, 2),
             (0, 3),
             (1, 2),(2,3),(1,3)]:
    plt.axhline(1.0, color="k", lw=1.0, alpha=0.5)
    plt.axvline(1.0, color="k", lw=1.0, alpha=0.5)
    plt.scatter(squared_feats[:, i], squared_feats[:, j], c=np.log10(refine_freqs), s=sizes)
    plt.loglog()
    plt.xlabel(f"scalar {i}")
    plt.ylabel(f"scalar {j}")
    plt.colorbar(label="log_10 frequency")
    plt.savefig(f"scatter_{i}_{j}.png")
    plt.show()

### running K-means

In [ ]:
import sklearn